# Tools: the manual round-trip

The smallest possible tool-use demo, done by hand so every step is visible.

1. Write a plain Python function (`get_current_datetime`).
2. Describe it to Claude as a JSON `ToolParam` schema.
3. Send a question with `tools=[...]` — Claude replies with a `tool_use` block instead of text.
4. Run the function yourself with the arguments Claude chose.
5. Append the result as a `tool_result` block and call the API again to get the final answer.

No loop, no helper machinery — just one request, one tool call, one follow-up.
See [21_tools_run_conversation.ipynb](21_tools_run_conversation.ipynb) for the automated version.


In [12]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [13]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [14]:
from datetime import datetime

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

In [15]:
# Default format: "2024-01-15 14:30:25"
get_current_datetime()

# Just hour and minute: "14:30"
#get_current_datetime("%H:%M")

'2026-08-20 17:22:24'

In [16]:
from anthropic.types import ToolParam


get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

In [17]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

messages.append({"role": "assistant", "content": response.content})

tool_result = get_current_datetime(**response.content[0].input)
f'Current time ({response.content[0].input}): {tool_result}'



"Current time ({'date_format': '%H:%M:%S'}): 17:22:25"

In [18]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": tool_result,
        "is_error": False
    }]
})

# messages


In [19]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response.content[0].text

'The exact time is **17:22:25** (5:22:25 PM).'